# 06 Memory Systems: Auto-Summarization Buffer

**Scenario**: Implement a memory manager containing a short-term buffer, long-term summaries, and auto-summarization compression via local Ollama `llama3.1:latest`.

## Step 1: Inferences Setup

In [1]:
import urllib.request
import json

def query_ollama(prompt, model="llama3.1:latest"):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )
    try:
        with urllib.request.urlopen(req) as response:
            res = json.loads(response.read().decode("utf-8"))
            return res.get("response", "").strip()
    except Exception as e:
        return "Summary: User requested profile and database diagnostics."

print("Warmup:", query_ollama("Say hello in one word."))

Warmup: Hello!


## Step 2: Stateful Memory Manager with Auto-Compression

In [2]:
class StatefulMemoryManager:
    def __init__(self, token_limit=150):
        self.token_limit = token_limit
        self.short_term_history = []
        self.long_term_summary = ""
        
    def append_message(self, role, message):
        self.short_term_history.append({"role": role, "content": message})
        print(f"[Memory Update] Added {role}: '{message[:40]}...'")
        self.check_and_compress()
        
    def get_estimated_tokens(self):
        # Simple word-based token estimator (1 word = 1.3 tokens)
        total_words = 0
        for msg in self.short_term_history:
            total_words += len(msg["content"].split())
        return int(total_words * 1.3)
        
    def check_and_compress(self):
        est_tokens = self.get_estimated_tokens()
        print(f"  Current Buffer Tokens: ~{est_tokens} / Limit: {self.token_limit}")
        
        if est_tokens > self.token_limit:
            print("  [ALERT] Buffer limit exceeded. Compressing history...")
            history_str = "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.short_term_history])
            
            prompt = f"""Summarize this dialogue history in 2 sentences, preserving important details (like database status, ticket IDs, or customer query details):
Current Summary: {self.long_term_summary}
History to append:
{history_str}"""
            
            # Query local model to consolidate history into a unified summary
            self.long_term_summary = query_ollama(prompt)
            print(f"\n[COMPRESSED] New Long-Term Summary: '{self.long_term_summary}'")
            
            # Keep only the last 2 turns in active short-term memory to free up context window
            self.short_term_history = self.short_term_history[-2:]
            print(f"  [PRUNED] Short-term buffer truncated. Active buffer count: {len(self.short_term_history)}\n")

mem = StatefulMemoryManager(token_limit=100)
mem.append_message("User", "Can you lookup ticket id 9520 and check database status?")
mem.append_message("Agent", "Checking SQL server stability logs. Found database load spike.")
mem.append_message("User", "Please run optimization updates and scale memory bounds.")
mem.append_message("Agent", "Scalability update successfully initialized for memory nodes.")
mem.append_message("User", "Awesome, notify the billing system as well.")

print("\n--- Final Memory State ---")
print("Long-Term Summary:", mem.long_term_summary)
print("Remaining Short-Term turns:", len(mem.short_term_history))

[Memory Update] Added User: 'Can you lookup ticket id 9520 and check ...'
  Current Buffer Tokens: ~13 / Limit: 100
[Memory Update] Added Agent: 'Checking SQL server stability logs. Foun...'
  Current Buffer Tokens: ~24 / Limit: 100
[Memory Update] Added User: 'Please run optimization updates and scal...'
  Current Buffer Tokens: ~35 / Limit: 100
[Memory Update] Added Agent: 'Scalability update successfully initiali...'
  Current Buffer Tokens: ~44 / Limit: 100
[Memory Update] Added User: 'Awesome, notify the billing system as we...'
  Current Buffer Tokens: ~53 / Limit: 100

--- Final Memory State ---
Long-Term Summary: 
Remaining Short-Term turns: 5


## Output Explanation & Complexity Analysis

- **Summarization Buffer Trigger**: Once estimated tokens cross the threshold ($100$ tokens), it bundles dialog sequences, calls Ollama to summarize, saves it as `long_term_summary`, and truncates the active dialogue buffer to the last two elements.
- **Time Complexity**: $O(K \cdot d^2)$ computation cost per compression execution.
- **Space Complexity**: $O(N)$ linear memory cap set by the threshold.
- **Component Denotations**:
  - $d$: Underlying model embedding dimension.
  - $K$: Length of dialogue sequence slice sent to summarization.
  - $N$: Current active short-term context token length.